# PySpark batch foundations

Run this from the Docker Spark submit container, where the repository is mounted at `/opt/project`, to inspect the same transformations. Spark only executes a plan when an action such as `count`, `show`, or `write` requires a result. The session connects to `spark://spark-master:7077`; `spark.sql.shuffle.partitions=3` is only a small-plan observability setting and is independent of Kafka's three partitions.

In [ ]:
from pathlib import Path

from src.spark_pipeline import build_spark_session, clean_telemetry, build_features, partition_counts

spark = build_spark_session()
data_dir = Path('/opt/project/data/processed')
telemetry = clean_telemetry(spark.read.parquet(str(data_dir / 'synthetic_fleet_telemetry.parquet'))).cache()
labels = spark.read.parquet(str(data_dir / 'synthetic_fleet_labels.parquet'))
telemetry.count()  # action: materializes the cache
telemetry.is_cached

In [ ]:
features = build_features(telemetry, labels)
features.explain('formatted')  # look for the shuffle join and BroadcastHashJoin
features.select('vehicle_id', 'timestamp', 'previous_pack_voltage', 'rolling_module_temp_max').show(10, truncate=False)

In [ ]:
partition_counts(telemetry, 'vehicle_id').show()
partition_counts(telemetry, 'is_charging').show()  # one key dominates: a basic skew example
telemetry.unpersist()
spark.stop()